In [2]:
import cvxpy as cp
import numpy as np
from itertools import product
from itertools import combinations
import time

In [15]:
def cal_pow(R, beta, h_val):
    M = len(R)
    order = np.argsort(np.array(beta) / np.array(h_val))
    P = np.zeros(M)
    for i in range(M):
        ui = order[i]
        sum_i_M = sum(R[order[k]] for k in range(i, M))
        sum_ip1_M = sum(R[order[k]] for k in range(i+1, M))
        P[ui] = (2**sum_i_M - 2**sum_ip1_M) / h_val[ui]
    return P

def is_duplicate(c, C, eps=1e-6):
    if len(C) == 0:
        return True 
    else:
        flag = 0
        for c_old in C:
            if np.max(np.abs(c - c_old)) < eps:
                flag = 1
        if flag:
            return False
        else:
            return True

In [31]:
n_user = 3
h_bad =  0.1
h_good = 1
pr_h_bad = 0.5
pr_h_good = (1- pr_h_bad)
r_max = 1
h_vec_u1 = np.array([h_bad,  h_good]).tolist()
permutations_list = []
permutations_list.extend(list(product(h_vec_u1, repeat=n_user )))
combined_h_vec = []
for permutation in permutations_list:
    arr = []
    for zz in permutation:
        arr.append(zz)
    combined_h_vec.append(arr)
p_h_u1 = np.array([pr_h_bad, pr_h_good])
permutations_list = []
permutations_list.extend(list(product(p_h_u1, repeat=n_user )))
pr_h = []
for permutation in permutations_list:
    arr = []
    for k in permutation:
        arr.append(k)
    pr_h.append(np.prod(arr))
hp = pr_h
h = combined_h_vec

In [33]:
lambda_k = n_user*[0.1]
v_k = n_user*[0.1]
alpha = 0.05
max_iter = 10000
mu_history = []

Pow1_history = []
Pow2_history = []

unique_Ps = []
unique_mus = []
unique_lambda = []
unique_pis = []

for itr in range(max_iter):
    # print('itr', itr)
    # print('dual var lambda', np.round( lambda_k, 6))
    # print('dual var vk', np.round( v_k, 3))
    B = lambda_k
    rho = np.arange(0, r_max+1, 1)
    P_bar_U = n_user*[5]
    W = list(product(rho, repeat=n_user))
    # W = [(0, 0, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1)]
    
    mu = np.zeros((len(h), len(W)))
    for k in range(len(h)):
        costs = np.zeros(len(W))
        for j in range(len(W)):
            for n in range(n_user):
                psi = 1 if W[j][n] > 0 else 0
                costs[j] += hp[k] * (
                    lambda_k[n] * cal_pow(W[j], B, h[k])[n]
                    - v_k[n] * psi)

        min_cost = np.min(costs)
        idx = np.where(np.abs(costs - min_cost) < 1e-8)[0]
        mu[k, idx] = 1.0 / len(idx)
    mu_final = mu
    # print('mu opt', mu_final)
    
    P1_temp = np.zeros((len(h), len(W)))
    P2_temp = np.zeros((len(h), len(W)))
    for k in range(len(h)):
        for j in range(len(W)):
            P1_temp[k, j] = cal_pow(W[j], B, h[k])[0] * mu_final[k][j]
            P2_temp[k, j] = cal_pow(W[j], B, h[k])[1] * mu_final[k][j]
    
    p = []
    for x in range(n_user):
        temp  = 0
        for k in range(len(h)):
            for j in range(len(W)):
                if W[j][x] != 0:
                    temp+= mu_final[k][j] * hp[k]
        p.append(temp)
        
    P_u = []
    for x in range(n_user):
        temp = 0
        for k in range(len(h)):
            for j in range( len(W)):
                temp+= cal_pow(W[j], B, h[k])[x] * mu_final[k][j] *  hp[k]
        P_u.append(temp)
    # print("Average powers:", np.round(P_u, 5))
    # print('Average pis:', np.round(p, 4))
  
           
    if itr > 5000:
        mu_history.append(mu_final.copy())
        Pow1_history.append(P1_temp.copy())
        Pow2_history.append(P2_temp.copy())
        # cz = np.array(P_u)
        # dz = np.array(p)
        # lz = np.array(lambda_k)
        # if is_duplicate(cz, unique_Ps):
        #     unique_Ps.append(cz)
        #     unique_mus.append(mu_final)
        #     unique_lambda.append(lz)
        #     unique_pis.append(dz)
  

    dual_objective = 0
    for n in range(n_user):
        dual_objective+= (np.sqrt(v_k[n]) - 1) + lambda_k[n] * (P_u[n] - P_bar_U[n]) - v_k[n]*(p[n]- 1/np.sqrt(v_k[n]))
    # print('dual objective:', dual_objective)

    subgradient_lambda = []
    subgradient_vk = []
    for n in range(n_user):
        subgradient_lambda.append(P_u[n] - P_bar_U[n])
        subgradient_vk.append(1/np.sqrt(v_k[n]) - p[n])
    eta_k = alpha / np.sqrt(itr + 1)
    for n in range(n_user):
        lambda_k[n] = max(0, lambda_k[n] + eta_k * subgradient_lambda[n])
        v_k[n] = max(0, v_k[n] + eta_k * subgradient_vk[n])
    # print('-------------------------------------')
print('--------------------------------------------------')
print('ERGODIC RESULTS')
T = len(mu_history)
mu_er = sum(mu_history) / T
Pow1_er = sum(Pow1_history)/T
Pow2_er = sum(Pow2_history)/T
print('final ergodic avg mu', np.round(mu_er, 3))
print('ergodic Pow 1', np.round(Pow1_er, 3))
print('ergodic Pow 2', np.round(Pow2_er, 3))
p_er = []
for n in range(n_user):
    temp = 0
    for k in range(len(h)):
        for j in range(len(W)):
            if W[j][n] > 0:
                temp += mu_er[k][j] * hp[k]
    p_er.append(temp)
age_bar = 0
pis_bar = []
for n in range(n_user):
    age_bar += (1 / p_er[n] - 1)
    pis_bar.append(p_er[n])
print('ergodic p:', np.round(pis_bar, 5))
print("Final average age from ergodic mu:", np.round(age_bar, 5))
# print('unique Ps', unique_Ps)
# print('unique lambda', unique_lambda)
# print('unique pis:', unique_pis)

--------------------------------------------------
ERGODIC RESULTS
final ergodic avg mu [[0.    0.018 0.019 0.306 0.012 0.324 0.32  0.   ]
 [0.    0.    0.    0.506 0.    0.494 0.    0.   ]
 [0.    0.    0.    0.498 0.    0.    0.502 0.   ]
 [0.    0.    0.    0.    0.    0.    0.    1.   ]
 [0.    0.    0.    0.    0.    0.504 0.496 0.   ]
 [0.    0.    0.    0.    0.    0.    0.    1.   ]
 [0.    0.    0.    0.    0.    0.    0.    1.   ]
 [0.    0.    0.    0.    0.    0.    0.    1.   ]]
ergodic Pow 1 [[ 0.     0.     0.     0.     0.116  4.965  4.637  0.   ]
 [ 0.     0.     0.     0.     0.     4.945  0.     0.   ]
 [ 0.     0.     0.     0.     0.     0.     5.021  0.   ]
 [ 0.     0.     0.     0.     0.     0.     0.    10.   ]
 [ 0.     0.     0.     0.     0.     1.008  0.992  0.   ]
 [ 0.     0.     0.     0.     0.     0.     0.     3.02 ]
 [ 0.     0.     0.     0.     0.     0.     0.     2.98 ]
 [ 0.     0.     0.     0.     0.     0.     0.     2.326]]
ergodic Pow 2 [[

In [11]:
print(len(unique_Ps), len(unique_pis))

0 0


In [251]:
P_temp = np.array(unique_Ps)
Mu_temp = np.array(unique_mus)
Lambda_temp = np.array(unique_lambda)
pis_temp = np.array(unique_pis)

L = P_temp.shape[0]

q = cp.Variable(L)
A_eq = np.vstack([P_temp.T, pis_temp.T, np.ones((1, L))])
b_eq = np.concatenate([P_bar_U, pis_bar, np.array([1.0])])

constraints = [ A_eq @ q == b_eq,
                q >= 0]

prob = cp.Problem(cp.Minimize(0), constraints)
prob.solve(solver=cp.SCS)

print(prob.status)
print("q =", np.round(q.value, 3))
print(np.sum(q.value))

import numpy as np

q_sol = q.value.copy()
q_sol[q_sol < 0] = 0   # remove tiny negatives
q_sol /= q_sol.sum()   # sum exactly 1

print("Sum(q) =", q_sol.sum())
q_val = q_sol

optimal
q = [0.061 0.072 0.083 0.001 0.068 0.056 0.088 0.01  0.067 0.068 0.082 0.006
 0.01  0.098 0.002 0.012 0.001 0.07  0.005 0.084 0.003 0.054]
1.0000374231991
Sum(q) = 0.9999999999999999


In [252]:
p_mu = pis_temp
p_mix = 0
for z in range(L):
    p_mix+= q_val[z] * np.array(p_mu[z])
J_mix = 0
for n in range(n_user):
    J_mix += (1 / p_mix[n] - 1)
print('Final age after randomized over policy:\n', np.round(J_mix, 7))
##########################################################
Pow_mu = P_temp
P_stack = np.vstack(Pow_mu).T 
P_mix = P_stack @ q_val
print("Final avg Power using randomized over policy + SIC Power:\n", P_mix)

Final age after randomized over policy:
 0.6090562
Final avg Power using randomized over policy + SIC Power:
 [4.99984659 4.9998479  4.99984694]
